# Notebook 0: Prepare Grouped Dataset For Training

This notebook prepares a user-provided class-root dataset for Notebook 2. The public default does not require access to the private research dataset.

Repo icinde hazir class-root datasetler: data/class_root_dataset/tomato_fruit, data/class_root_dataset/grape_fruit ve data/class_root_dataset/grape_leaf.

Audit hedefi sadece "temiz" veri degil, adapter performansi icin guvenilir veri uretmek olmalidir:
- exact duplicate ve near-duplicate ailelerini birlikte tutun.
- ayni capture/session/source ailelerini ayirmayin; source leakage'i engelleyin.
- gercek hard negatives ve OOD pool'u klasification siniflarindan ayri yonetin.
- sadece audit raporu temiz oldugu icin degil, class support ve source dagilimi guvenilir oldugu icin materyalize edin.

Public usage:
1. Leave `DATASET_RELEASE_TAG` empty.
2. Upload or mount your class-root dataset and set `REPO_DATASET_ROOT` or `DATASET_ROOT`.
3. Run the audit and materialization cells in order.
4. For a zero-data smoke run, use Notebook 2's `public_sample` default instead.

Maintainer local audit akisi gerektiginde `DATASET_RELEASE_TAG` bos birakilarak kullanilir.


## Hizli Akis

Varsayilan akis, sabit GitHub Release'i indirip manifest/hash dogrulamasi sonrasi Notebook 2'nin tuketecegi `continual/`, `val/`, `test/`, `ood/`, `oe/` yapisini cache altinda materyalize eder.

- Musteri akisinda yalnizca sabit repository/tag/target/cache parametrelerini kullanin.
- Notebook yazma tokeni istemez ve dataset commit/push islemi yapmaz.
- Token veya ag hatasi `DATASET_RELEASE_ACCESS_BLOCKER`; hash/manifest uyusmazligi ise ayri integrity hatasidir.
- Local class-root audit akisi yalnizca release tag bosken devreye girer.


## Sabit GitHub Release Secimi

The private immutable Release path below is maintainer-only. Public users should leave the tag empty and provide their own data.

### Nasil Kullanilir

1. `DATASET_RELEASE_REPOSITORY` ve immutable `DATASET_RELEASE_TAG` kimligini belirtin.
2. `DATASET_RELEASE_TARGET` ile `<crop>__<part>` hedefini secin.
3. Cache root degerini belirleyin ve salt-okunur tokeni Colab secret olarak ekleyin.
4. Erisim kontrolu ve materyalizasyon hucrelerini sirayla calistirin.

### Ornek

```python
DATASET_RELEASE_REPOSITORY = "EfeErim/bitirmeprojesi"
DATASET_RELEASE_TAG = "aads-dataset-v1.0.0"
DATASET_RELEASE_TARGET = "tomato__leaf"
```

Bu akis Drive'a ve dataset Git commitlerine bagimli degildir; release assetleri manifest ve SHA-256 ile dogrulanir.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def _configure_colab_git_read_access():
    token = str(os.environ.get('AADS_GITHUB_RELEASE_READ_TOKEN', '')).strip()
    if not token:
        try:
            from google.colab import userdata
            token = str(userdata.get('AADS_GITHUB_RELEASE_READ_TOKEN') or '').strip()
        except Exception:  # Colab secret access raises provider-specific exceptions.
            token = ''
    os.environ['GIT_TERMINAL_PROMPT'] = '0'
    if not token:
        return
    askpass = Path('/tmp/aads_git_read_askpass.sh')
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        "*Username*) printf '%s\\n' 'x-access-token' ;;\n"
        "*) printf '%s\\n' \"$AADS_GIT_READ_TOKEN\" ;;\n"
        'esac\n',
        encoding='utf-8',
    )
    askpass.chmod(0o700)
    os.environ['GIT_ASKPASS'] = str(askpass)
    os.environ['GIT_ASKPASS_REQUIRE'] = 'force'
    os.environ['AADS_GIT_READ_TOKEN'] = token
    os.environ['AADS_GITHUB_RELEASE_READ_TOKEN'] = token


_configure_colab_git_read_access()

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')


def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root

    if not CLONE_TARGET.exists():
        CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
        completed = subprocess.run(
            ['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)],
            check=False,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
        if completed.stdout:
            print(completed.stdout)
        if completed.returncode != 0:
            raise RuntimeError(
                'Notebook 0 repo bootstrap failed. Set AADS_REPO_ROOT to an existing checkout '
                'or provide the read-only AADS_GITHUB_RELEASE_READ_TOKEN Colab secret.'
            )

    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET


_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb0_cell02_bootstrap.py', globals())

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb0_cell03_runtime_setup.py', globals())


In [ ]:
with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    # --- Notebook 0 parametreleri ---
    # Once bu hucreyi duzenleyin, sonra alttaki hucreleri sirayla calistirin.

    # --- HIZLI KULLANIM: Salt-okunur release icin bu dort degeri sabit tutun/degistirin ---
    DATASET_RELEASE_REPOSITORY = "EfeErim/bitirmeprojesi"
    DATASET_RELEASE_TAG = ""
    DATASET_RELEASE_TARGET = "tomato__leaf"
    DATASET_RELEASE_CACHE_ROOT = ".runtime_tmp/dataset_release_cache"

    # DATASET_RELEASE_TAG bos birakilirsa asagidaki maintainer audit/materialize yolu kullanilir.
    # REPO_DATASET_ROOT: repo ici relatif yol veya mutlak yol olabilir.
    REPO_DATASET_ROOT = "data/class_root_dataset"  # replace with your own class-root path

    # REPO_DATASET_NAME: parent klasor verdiyseniz dataset adi; dogrudan dataset yolu verdiyseniz bos birakin.
    REPO_DATASET_NAME = ""

    # DATASET_ROOT: opsiyonel override. Bos ise yukaridaki secim kullanilir.
    DATASET_ROOT = ""

    # IMPORT_FROM_DRIVE: True ise Drive path doluyken Drive seceneklerini sorar; varsayilan akis repo-icidir.
    IMPORT_FROM_DRIVE = False

    # DRIVE_DATASET_PATH: Opsiyonel Drive dataset parent yolu. Bos ise repo data klasoru kullanilir.
    DRIVE_DATASET_PATH = ""

    # DRIVE_DATASET_NAME: Kopyalanacak klasor adi. Bos ise secenekleri listeler ve sorar.
    DRIVE_DATASET_NAME = ""

    # CROP_NAME / PART_NAME: bos birakirsan notebook sorar.
    CROP_NAME = ""
    PART_NAME = ""

    # AUTORUN_AUDIT: True ise audit hucreleri varsayilan akista calisir.
    AUTORUN_AUDIT = True

    # AUTORUN_MATERIALIZE: True ise runtime dataset materyalizasyonu etkin olur.
    AUTORUN_MATERIALIZE = False

    run_id = RUN_ID
    STATE = {
        "validated": False,
        "audit_summary": None,
        "artifact_root": None,
        "runtime_dataset_root": None,
        "dataset_root": None,
        "dataset_name": None,
        "dataset_source": None,
        "selected_ood_dataset_name": None,
        "resolved_ood_root": None,
        "prep_materialization_result": None,
        "access_report": None,
        "dataset_release_report": None,
    }
    print(
        f"[PARAMS] run_id={run_id} crop={CROP_NAME or '(ask)'} part={PART_NAME or '(ask)'} "
        f"repo_dataset_root={REPO_DATASET_ROOT} repo_dataset_name={REPO_DATASET_NAME or '(ask)'} "
        f"dataset_root_override={DATASET_ROOT or '(none)'}"
    )
    print(
        f"[DRIVE] import_from_drive={IMPORT_FROM_DRIVE} drive_path={DRIVE_DATASET_PATH or '<none>'} "
        f"drive_name={DRIVE_DATASET_NAME or '<ask>'}"
    )
    print(
        f"[OOD] repo_root={OOD_DATASET_ROOT} repo_name={OOD_DATASET_NAME or '<none>'} direct_root={OOD_ROOT or '<none>'}"
    )
    print(
        f"[DATASET RELEASE] repository={DATASET_RELEASE_REPOSITORY} tag={DATASET_RELEASE_TAG or '<local-audit>'} "
        f"target={DATASET_RELEASE_TARGET or '<none>'} cache_root={DATASET_RELEASE_CACHE_ROOT}"
    )
    print(
        "[SONRAKI] Erisim kontrolu -> audit -> guided/00_start_here.md incelemesi -> "
        f"materialize={MATERIALIZE_AFTER_REVIEW} akisi. Runtime cikti: {PREPARED_RUNTIME_ROOT}"
    )
    PREPARE_DATASET_FROM_REPORTS = PREPARE_DATASET_FROM_REPORTS and not bool(DATASET_RELEASE_TAG)
    MATERIALIZE_AFTER_REVIEW = MATERIALIZE_AFTER_REVIEW and not bool(DATASET_RELEASE_TAG)


In [ ]:
with TELEMETRY.capture_cell_output("Cell 3a: Google Drive Dataset Import (Opsiyonel)"):
    from scripts.colab_dataset_layout import resolve_dataset_directory_from_parent
    from scripts.colab_repo_bootstrap import mount_drive_if_available
    import shutil
    
    def import_dataset_from_drive(
        source_path: str | Path,
        destination_path: str | Path,
        dataset_name: str,
        overwrite: bool = False
    ) -> bool:
        """Drive'dan dataseti yerel klasore kopyala."""
        source_path = Path(source_path)
        destination_path = Path(destination_path).expanduser()
        destination_path.mkdir(parents=True, exist_ok=True)
        
        target = destination_path / dataset_name
        
        if target.exists():
            if not overwrite:
                print(f"[DRIVE] Hedef klasor zaten var: {target}")
                return False
            shutil.rmtree(target)
            print(f"[DRIVE] Mevcut klasor silindi: {target}")
        
        try:
            print(f"[DRIVE] Kopyalama baslatildi: {source_path} -> {target}")
            shutil.copytree(source_path, target, dirs_exist_ok=False)
            print(f"[DRIVE] Basariyla kopyalandi: {target}")
            return True
        except Exception as e:
            print(f"[DRIVE] Kopyalama basarisiz: {e}")
            return False
    
    def _drive_destination_parent() -> Path:
        explicit_root = str(DATASET_ROOT).strip()
        if explicit_root:
            root_path = Path(explicit_root).expanduser()
            return root_path.resolve() if root_path.is_absolute() else (ROOT / root_path).resolve()
        return ROOT / "data" / "imported_from_drive"
    
    # Google Drive'dan dataset alma secenegi (parametrelerden al)
    if IMPORT_FROM_DRIVE and str(DRIVE_DATASET_PATH).strip():
        print("[DRIVE] Google Drive icinde dataset ara...")
        mount_drive_if_available()
        
        drive_path = Path(DRIVE_DATASET_PATH).expanduser()
        if not drive_path.exists():
            raise RuntimeError(f"Drive yolu bulunamadi: {drive_path}")
        dataset_name, source, available_datasets = resolve_dataset_directory_from_parent(
            dataset_parent=drive_path,
            requested_name=DRIVE_DATASET_NAME,
            prompt_label="Drive dataset",
        )
        print(f"[DRIVE] secilebilir datasetler={available_datasets}")
        dest_parent = _drive_destination_parent()
        if import_dataset_from_drive(source, dest_parent, dataset_name):
            DATASET_ROOT = str(dest_parent / dataset_name)
            print(f"[DRIVE] DATASET_ROOT guncellendi: {DATASET_ROOT}")
        else:
            existing_target = dest_parent / dataset_name
            if existing_target.is_dir():
                DATASET_ROOT = str(existing_target)
                print(f"[DRIVE] Mevcut kopya kullaniliyor, DATASET_ROOT guncellendi: {DATASET_ROOT}")
    else:
        print("[DRIVE] IMPORT_FROM_DRIVE=False veya DRIVE_DATASET_PATH bos. Drive import atlanacak.")

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb0_cell06_access_check.py', globals())


In [ ]:
with TELEMETRY.capture_cell_output("Cell 4: Dataset Audit"):
    from scripts.notebook_helpers.nb0_grouped_dataset_prep_helpers import run_dataset_audit

    if DATASET_RELEASE_TAG:
        STATE["validated"] = True
        STATE["audit_summary"] = {"runtime_ready": True, "source": "github_release"}
        print("[DATASET RELEASE] Local class-root audit skipped; immutable release verification is authoritative.")
    else:
        STATE, CROP_NAME, PART_NAME = run_dataset_audit(
            ROOT=ROOT,
            STATE=STATE,
            TELEMETRY=TELEMETRY,
            DEVICE=DEVICE,
            EMBEDDING_BATCH_SIZE=EMBEDDING_BATCH_SIZE,
            NEIGHBORS=NEIGHBORS,
            REPO_DATASET_ROOT=REPO_DATASET_ROOT,
            REPO_DATASET_NAME=REPO_DATASET_NAME,
            DATASET_ROOT=DATASET_ROOT,
            CROP_NAME=CROP_NAME,
            PART_NAME=PART_NAME,
            PREP_ARTIFACT_ROOT=PREP_ARTIFACT_ROOT,
            PREP_DINOV3_MODEL_ID=PREP_DINOV3_MODEL_ID,
            PREP_BIOCLIP_MODEL_ID=PREP_BIOCLIP_MODEL_ID,
            UNDER_MIN_EVAL_POLICY=UNDER_MIN_EVAL_POLICY,
            IMPORT_FROM_DRIVE=IMPORT_FROM_DRIVE,
            DRIVE_DATASET_PATH=DRIVE_DATASET_PATH,
            INTERACTIVE_AUDIT_REVIEW=INTERACTIVE_AUDIT_REVIEW,
            MAX_INTERACTIVE_REVIEW_ITEMS=MAX_INTERACTIVE_REVIEW_ITEMS,
        )


In [ ]:
if "TELEMETRY" not in globals() or "STATE" not in globals():
    raise RuntimeError("Runtime setup hazir degil. Once bootstrap ve Cell 3 runtime setup hucrelerini, sonra Cell 4 Dataset Audit hucrelerini sirayla calistirin.")

with TELEMETRY.capture_cell_output("Cell 4b: Prepare Dataset For Materialization"):
    from scripts.prepare_grouped_runtime_dataset import build_human_review_packet, build_prepared_dataset_key, format_human_review_packet
    from scripts.prepare_materialization_dataset import prepare_class_root_for_materialization
    from src.shared.json_utils import write_json

    if not STATE.get("validated") or STATE.get("audit_summary") is None:
        raise RuntimeError("Once dataset audit hucresini calistirin.")

    summary = STATE["audit_summary"]
    if not PREPARE_DATASET_FROM_REPORTS:
        print("[PREP] PREPARE_DATASET_FROM_REPORTS=False. Audit raporlari pasif birakildi.")
    else:
        dataset_key = build_prepared_dataset_key(CROP_NAME, PART_NAME)
        prepared_class_root_parent = Path(PREPARED_CLASS_ROOT).expanduser()
        if not prepared_class_root_parent.is_absolute():
            prepared_class_root_parent = (ROOT / prepared_class_root_parent).resolve()
        prepared_class_root = prepared_class_root_parent / dataset_key
        prepared_artifact_root = STATE["artifact_root"].parent / f"{STATE['artifact_root'].name}_prepared"
        prep_counts = dict(summary.get("summary", {}))
        print("[PREP] Rapor tabanli hazirlik ozeti:")
        print(
            f"  dataset_key={dataset_key} total_images={prep_counts.get('total_images', 0)} "
            f"cross_class_conflicts={prep_counts.get('cross_class_conflicts', 0)} "
            f"same_class_high_risk_clusters={prep_counts.get('same_class_high_risk_clusters', 0)}"
        )
        print(f"  source_dataset_root={STATE['dataset_root']}")
        print(f"  source_artifact_root={STATE['artifact_root']}")
        print(f"  prepared_class_root={prepared_class_root}")
        print(f"  prepared_artifact_root={prepared_artifact_root}")
        print(f"  cleanup_seed={CLEANUP_SEED}")
        prep_result = prepare_class_root_for_materialization(
            class_root=STATE["dataset_root"],
            crop_name=CROP_NAME,
            part_name=PART_NAME,
            audit_artifact_root=STATE["artifact_root"],
            prepared_class_root=prepared_class_root,
            prepared_artifact_root=prepared_artifact_root,
            taxonomy_path=ROOT / "config" / "plant_taxonomy.json",
            dino_model_id=PREP_DINOV3_MODEL_ID,
            bioclip_model_id=PREP_BIOCLIP_MODEL_ID,
            device=DEVICE,
            batch_size=EMBEDDING_BATCH_SIZE,
            neighbors=NEIGHBORS,
            cleanup_seed=CLEANUP_SEED,
            quarantine_cross_class_conflicts=True,
            under_min_eval_policy=UNDER_MIN_EVAL_POLICY,
            materialization_strategy="auto",
            progress_fn=lambda message: print(f"[PREP] {message}"),
        )
        STATE["prep_materialization_result"] = prep_result
        STATE["dataset_root"] = Path(prep_result["prepared_class_root"])
        STATE["dataset_source"] = "prepared_class_root"
        STATE["artifact_root"] = Path(prep_result["prepared_artifact_root"])
        STATE["audit_summary"] = prep_result["rerun_summary"]
        summary = STATE["audit_summary"]
        review_packet = build_human_review_packet(
            summary,
            artifact_root=STATE["artifact_root"],
            max_review_items=MAX_INTERACTIVE_REVIEW_ITEMS,
        )
        STATE["human_review_packet"] = review_packet
        STATE["human_review_approved"] = not bool(review_packet.get("pause_recommended"))
        write_json(STATE["artifact_root"] / "human_review_packet.json", review_packet, ensure_ascii=False)
        print(
            f"[PREP] prepared_runtime_ready={prep_result.get('prepared_runtime_ready')} "
            f"dataset_key={prep_result.get('dataset_key')} prepared_class_root={STATE['dataset_root']}"
        )
        print(f"[PREP] Hazirlik sonrasi artifact_root={STATE['artifact_root']}")
        print(format_human_review_packet(review_packet))
        if INTERACTIVE_AUDIT_REVIEW and review_packet.get("pause_recommended"):
            print("[PREP] Prepared audit gate: Enter guvenli varsayilani uygular.")
            if not _prompt_yes_no("Hazirlanmis dataset ile devam edilsin mi?", True):
                STATE["human_review_stop_requested"] = True
                STATE["human_review_approved"] = False
                raise RuntimeError("Human review tarafindan durduruldu. Artifactleri inceleyip hucreyi yeniden calistirin.")
            STATE["human_review_stop_requested"] = False
            STATE["human_review_approved"] = True
        else:
            STATE["human_review_stop_requested"] = False
            STATE["human_review_approved"] = True
        TELEMETRY.update_latest(
            {
                "phase": "data_prep_prepared",
                "dataset_root": str(STATE["dataset_root"]),
                "dataset_source": str(STATE.get("dataset_source") or "prepared_class_root"),
                "artifact_root": str(STATE["artifact_root"]),
                "runtime_ready": bool(summary.get("runtime_ready")),
            }
        )


In [ ]:
with TELEMETRY.capture_cell_output("Cell 5: Materialize Runtime Dataset"):
    from scripts.notebook_helpers.nb0_grouped_dataset_prep_helpers import run_materialize_runtime_dataset

    run_materialize_runtime_dataset(
        ROOT=ROOT,
        STATE=STATE,
        TELEMETRY=TELEMETRY,
        CROP_NAME=CROP_NAME,
        PART_NAME=PART_NAME,
        OOD_ROOT=OOD_ROOT,
        OOD_DATASET_NAME=OOD_DATASET_NAME,
        OOD_DATASET_ROOT=OOD_DATASET_ROOT,
        ASK_FOR_OOD_ROOT=ASK_FOR_OOD_ROOT,
        PREPARED_RUNTIME_ROOT=PREPARED_RUNTIME_ROOT,
        MATERIALIZE_AFTER_REVIEW=MATERIALIZE_AFTER_REVIEW,
        DATASET_RELEASE_REPOSITORY=DATASET_RELEASE_REPOSITORY,
        DATASET_RELEASE_TAG=DATASET_RELEASE_TAG,
        DATASET_RELEASE_TARGET=DATASET_RELEASE_TARGET,
        DATASET_RELEASE_CACHE_ROOT=DATASET_RELEASE_CACHE_ROOT,
        REPO_NOTEBOOK_OUTPUT_PATH=REPO_NOTEBOOK_OUTPUT_PATH,
        REPO_RUN_DIR=REPO_RUN_DIR,
        REPO_RUN_EXPORTS=REPO_RUN_EXPORTS,
        INTERACTIVE_AUDIT_REVIEW=INTERACTIVE_AUDIT_REVIEW,
    )
